## 09. Extensión del Analista: Texto Narrativo

> **Objetivo:** Validar si el LLM puede extraer conceptos-valor en bloques de **texto narrativo** (frases con cifras incrustadas) con el mismo rigor que en tablas, garantizando cero alucinaciones y sin omitir datos cuando coexisten varias cifras en la misma oración.

---

### 📌 Notas de desarrollo
* **Aislamiento:** Las pruebas se realizan de forma independiente.
* **Estado del agente:** La función `agente_analista` **no** se modifica en esta fase.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

### Obtener un bloque de texto real, sin gastar cuota

In [ ]:
from src_agents.rag.extractor_generico import extraer_documento

bloques_docx = extraer_documento(DATA_DIR / "financiero_informe_1q.docx")
bloques_texto = [b for b in bloques_docx if b.tipo_bloque == "texto"]

print(f"Bloques de texto encontrados: {len(bloques_texto)}")
for b in bloques_texto[:3]:
    print(f"--- {b.etiqueta} ---")
    print(b.contenido[:200])
    print()

### 📝 Diseño del Prompt

A diferencia del enfoque para tablas, en este caso **no existen cabeceras** que sirvan de guía. El prompt está optimizado para:

* **Extracción en prosa:** Identificar cifras dentro de texto narrativo y asignarles un concepto descriptivo propio.
* **Cobertura completa:** Instrucción explícita para capturar **todas** las cifras de una oración cuando coexistan varias en la misma frase.
* **Manejo de bloques vacíos:** Retornar `"ninguna"` cuando el bloque carezca de datos numéricos (ej. bloques de firma o cierre).

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

PROMPT_TEXTO = """Recibes un fragmento de texto de un documento municipal.
Puede contener cifras relevantes (cantidades, indicadores, totales) o
ninguna en absoluto (texto introductorio, firmas, cierres formales).

Tu tarea: extrae TODAS las cifras relevantes que encuentres, con un
nombre de concepto claro y corto para cada una. Si una frase menciona
varias cifras distintas, extráelas TODAS por separado, no solo la
primera.

Si el texto no contiene ninguna cifra relevante, responde exactamente:
SIN_DATOS

Si contiene cifras, responde en este formato, una línea por cifra:
concepto: valor

Texto:
{texto}"""

_plantilla_texto = ChatPromptTemplate.from_messages([("human", PROMPT_TEXTO)])

modelo_texto = ChatGroq(
    model=os.environ.get("GROQ_MODEL_ANALISTA", "llama-3.3-70b-versatile"),
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

### Probamos SOLO con el bloque que tiene datos reales

In [ ]:
bloque_con_datos = bloques_texto[1]  # "DATOS SOLICITADOS", el que tiene cifras
respuesta = modelo_texto.invoke(_plantilla_texto.invoke({"texto": bloque_con_datos.contenido}))
print(respuesta.content)

### Prueba con un caso sin datos para comprobar si se lo inventa.

In [ ]:
bloque_sin_datos = bloques_texto[0]  # el párrafo introductorio, "ASUNTO: INFORME..."
respuesta_vacia = modelo_texto.invoke(_plantilla_texto.invoke({"texto": bloque_sin_datos.contenido}))
print(respuesta_vacia.content)

### Arreglamos un problema de estilo en los formatos. Ajuste del prompt

In [ ]:
PROMPT_TEXTO = """Recibes un fragmento de texto de un documento municipal.
Puede contener cifras relevantes (cantidades, indicadores, totales) o
ninguna en absoluto (texto introductorio, firmas, cierres formales).

Tu tarea: extrae TODAS las cifras relevantes que encuentres, con un
nombre de concepto claro, en español natural, con mayúsculas y
espacios normales (ej. "Solicitantes bolsa de empleo", NO
"solicitantes_bolsa_empleo"). Si una frase menciona varias cifras
distintas, extráelas TODAS por separado, no solo la primera.

Si el texto no contiene ninguna cifra relevante, responde exactamente:
SIN_DATOS

Si contiene cifras, responde en este formato, una línea por cifra:
concepto: valor

Texto:
{texto}"""

In [ ]:
print(repr(respuesta.content))

In [ ]:
print(_plantilla_texto.messages[0].prompt.template[:200])

### Arreglamos esto con snake_case

In [ ]:
_plantilla_texto = ChatPromptTemplate.from_messages([("human", PROMPT_TEXTO)])

In [ ]:
bloque_con_datos = bloques_texto[1]
respuesta = modelo_texto.invoke(_plantilla_texto.invoke({"texto": bloque_con_datos.contenido}))
print(respuesta.content)

### El parser tiene que ser tolerante a esto, no confiar en que el modelo obedezca siempre

In [ ]:
import re
from src_agents.models.state import ConceptoValor

def _parsear_respuesta_texto(texto: str, etiqueta: str, fuente: str) -> list[ConceptoValor]:
    """Convierte la respuesta del modelo (texto narrativo) en ConceptoValor.
    Tolerante a explicaciones entre parentesis que el modelo a veces
    añade pese a la instruccion de no hacerlo (ej. '0 (no se proporciona
    un valor especifico...)') -- se descarta esa parte, no se confia en
    que el modelo obedezca el formato al pie de la letra."""
    if texto.strip() == "SIN_DATOS":
        return []

    conceptos = []
    for linea in texto.splitlines():
        linea = linea.strip()
        if not linea or ":" not in linea:
            continue
        concepto, _, valor = linea.partition(":")
        concepto, valor = concepto.strip(), valor.strip()

        # Descarta cualquier explicación entre paréntesis pegada al valor
        valor = re.sub(r"\s*\(.*?\)\s*$", "", valor).strip()
        if not valor:
            continue

        conceptos.append(ConceptoValor(concepto=concepto, valor=valor, fuente=f"{fuente} - {etiqueta}"))
    return conceptos

In [ ]:
conceptos_prueba = _parsear_respuesta_texto(respuesta.content, bloque_con_datos.etiqueta, bloque_con_datos.fuente)
print(f"Conceptos parseados: {len(conceptos_prueba)}")
for c in conceptos_prueba[:3]:
    print(c)

### 27 conceptos parseados limpiamente, sin ningún paréntesis colado, y trazabilidad correcta con la etiqueta real del bloque. El parser aguanta la desobediencia del modelo tal como esperábamos. Con esto validado, integramos.

### Integrar sin romper lo que ya funciona

agente_analista ya filtra bloques por tipo_bloque == "tabla". Añadimos
un segundo bucle para tipo_bloque == "texto", con su propio prompt y
parser, y se juntan los resultados en la misma lista de ConceptoValor.
No se toca la lógica de tablas para nada.

### Prueba con el doc compelto

In [ ]:
from src_agents.agents.analyst import agente_analista
from src_agents.rag.extractor_generico import extraer_documento

bloques_completos = extraer_documento(DATA_DIR / "financiero_informe_1q.docx")
resultado_analista = agente_analista({"documents": bloques_completos})
print(f"Total conceptos: {len(resultado_analista['analysis'].datos)}")

In [ ]:
conceptos_de_tabla = [c for c in resultado_analista["analysis"].datos if "(" in c.concepto and ")" in c.concepto]
conceptos_de_texto = [c for c in resultado_analista["analysis"].datos if c not in conceptos_de_tabla]

print(f"De tablas: {len(conceptos_de_tabla)}")
print(f"De texto narrativo: {len(conceptos_de_texto)}")